In [7]:
import numpy as np
import cv2 as cv
import time
import matplotlib.pyplot as plt
from tkinter import *
import tkinter as tk
from PIL import Image, ImageTk
import random   

window = tk.Tk()

# MAIN WINDOW SETTINGS
imgs = tk.PhotoImage(file="C:\\Users\\HP\\Documents\\Object Detection\\Project\\Object Detection In Real Time\\finalize.png")

window.title("OBJECT DETECTION")
window.geometry('1100x650')
window.configure(background='black')

LabelImg = Label(image=imgs)
LabelImg.pack()

message = tk.Label(window, text="OBJECT DETECTION", bg="light blue", fg="black", 
                   width=48, height=2, font=('times', 30, 'italic bold'))
message.place(x=50, y=10)


# --------------------------------------------------------------------
# SAFE WINDOW CHECK (PREVENTS CRASH)
# --------------------------------------------------------------------
def window_alive():
    try:
        return window.winfo_exists() == 1
    except:
        return False


# --------------------------------------------------------------------
# EXIT FUNCTION
# --------------------------------------------------------------------
def exit():
    try:
        cap.release()
    except:
        pass
    
    if window_alive():
        window.destroy()


# --------------------------------------------------------------------
# RUN ON IMAGE
# --------------------------------------------------------------------
def runToTheImage():

    if not window_alive():
        return

    labels = []
    classFile = 'coco.names'
    with open(classFile, 'rt') as f:
        labels = f.read().rstrip('\n').split('\n')

    label_colors = np.random.uniform(0, 255, (len(labels), 3))

    tf_net = cv.dnn.readNetFromTensorflow(
        "frozen_inference_graph.pb",
        "ssd_mobilenet_v3_large_coco_2020_01_14.pbtxt"
    )

    images = [
        "person.jpg", "cats.jpg", "dog.jpg", "electro.png",
        "reading.jpg", "sofa.jpg", "street.jpg", "bus.jpg","Hand.jpg","Finger.jpg","pen.jpg"
    ]

    number = random.randint(0, len(images)-1)
    img = cv.imread(images[number])

    rows, cols = img.shape[:2]

    blob = cv.dnn.blobFromImage(img, 1.0/127.5, (300, 300),
                                (127.5, 127.5, 127.5), swapRB=True, crop=False)
    tf_net.setInput(blob)
    out = tf_net.forward()

    confidence = 0.5

    for detection in out[0, 0, :, :]:
        score = float(detection[2])
        if score > confidence:
            label = int(detection[1]) - 1
            left = int(detection[3] * cols)
            top = int(detection[4] * rows)
            right = int(detection[5] * cols)
            bottom = int(detection[6] * rows)

            text = f"{labels[label]} - {round(score*100, 2)}%"
            cv.putText(img, text, (left, top), cv.FONT_HERSHEY_SIMPLEX,
                       0.5, label_colors[label], 2)
            cv.rectangle(img, (left, top), (right, bottom),
                         label_colors[label], 3)

    if not window_alive():
        return

    imageFrame = Frame(window, bg="grey", width=700, height=600)
    rst = tk.Label(imageFrame, background="snow", fg="black", font=("", 15))

    out_img = cv.resize(img, (600, 500))
    rgb = cv.cvtColor(out_img, cv.COLOR_BGR2RGB)
    imgTk = ImageTk.PhotoImage(Image.fromarray(rgb))

    rst.config(image=imgTk)
    rst.image = imgTk
    rst.place(x=50, y=40)

    imageFrame.place(x=250, y=120)

    def clear():
        imageFrame.destroy()

    Button(window, text="Clear", command=clear, fg="black", bg="lawn green",
           padx=10, pady=10, width=10, height=2, font=('times', 15, 'bold')).place(x=1000, y=350)


# --------------------------------------------------------------------
# RUN ON VIDEO
# --------------------------------------------------------------------
def runToTheVideo():

    if not window_alive():
        return

    labels = []
    classFile = 'coco.names'
    with open(classFile, 'rt') as f:
        labels = f.read().rstrip('\n').split('\n')

    label_colors = np.random.uniform(0, 255, (len(labels), 3))

    tf_net = cv.dnn.readNetFromTensorflow(
        "frozen_inference_graph.pb",
        "ssd_mobilenet_v3_large_coco_2020_01_14.pbtxt"
    )

    cap = cv.VideoCapture("video1.mp4")
    pause = True

    while pause:

        if not window_alive():
            cap.release()
            return

        succ, img = cap.read()
        if not succ:
            break

        img = cv.flip(img, 1)

        rows, cols = img.shape[:2]

        blob = cv.dnn.blobFromImage(img, 1.0/127.5, (300, 300),
                                    (127.5, 127.5, 127.5), swapRB=True, crop=False)

        tf_net.setInput(blob)
        out = tf_net.forward()

        confidence = 0.5

        for detection in out[0, 0, :, :]:
            score = float(detection[2])
            if score > confidence:
                label = int(detection[1]) - 1

                left = int(detection[3] * cols)
                top = int(detection[4] * rows)
                right = int(detection[5] * cols)
                bottom = int(detection[6] * rows)

                text = f"{labels[label]} - {round(score*100, 2)}%"
                cv.putText(img, text, (left, top), cv.FONT_HERSHEY_SIMPLEX,
                           0.6, label_colors[label], 2)
                cv.rectangle(img, (left, top), (right, bottom),
                             label_colors[label], 2)

        if not window_alive():
            cap.release()
            return

        imageFrame = Frame(window, bg="sky blue", width=700, height=600)
        rst = tk.Label(imageFrame, background="snow", fg="black")

        out_img = cv.resize(img, (600, 500))
        rgb = cv.cvtColor(out_img, cv.COLOR_BGR2RGB)
        imgTk = ImageTk.PhotoImage(Image.fromarray(rgb))

        rst.config(image=imgTk)
        rst.image = imgTk
        rst.place(x=50, y=40)
        imageFrame.place(x=250, y=120)

        try:
            window.update()
        except:
            break

    cap.release()
    cv.destroyAllWindows()


# --------------------------------------------------------------------
# RUN LIVE CAMERA
# --------------------------------------------------------------------
def runLive():

    if not window_alive():
        return

    labels = []
    classFile = 'coco.names'
    with open(classFile, 'rt') as f:
        labels = f.read().rstrip('\n').split('\n')

    label_colors = np.random.uniform(0, 255, (len(labels), 3))

    tf_net = cv.dnn.readNetFromTensorflow(
        "frozen_inference_graph.pb",
        "ssd_mobilenet_v3_large_coco_2020_01_14.pbtxt"
    )

    cap = cv.VideoCapture(0)
    pause = True

    while pause:

        if not window_alive():
            cap.release()
            return

        succ, img = cap.read()
        if not succ:
            break

        img = cv.flip(img, 1)
        rows, cols = img.shape[:2]

        blob = cv.dnn.blobFromImage(img, 1.0/127.5, (300, 300),
                                    (127.5, 127.5, 127.5), swapRB=True, crop=False)
        tf_net.setInput(blob)
        out = tf_net.forward()

        confidence = 0.7

        for detection in out[0, 0, :, :]:
            score = float(detection[2])
            if score > confidence:
                label = int(detection[1]) - 1

                left = int(detection[3] * cols)
                top = int(detection[4] * rows)
                right = int(detection[5] * cols)
                bottom = int(detection[6] * rows)

                text = f"{labels[label]} - {round(score*100, 2)}%"
                cv.putText(img, text, (left, top), cv.FONT_HERSHEY_SIMPLEX,
                           0.5, label_colors[label], 2)
                cv.rectangle(img, (left, top), (right, bottom),
                             label_colors[label], 3)

        if not window_alive():
            cap.release()
            return

        imageFrame = Frame(window, bg="sky blue", width=700, height=600)
        rst = tk.Label(imageFrame, background="snow", fg="black")

        out_img = cv.resize(img, (600, 500))
        rgb = cv.cvtColor(out_img, cv.COLOR_BGR2RGB)
        imgTk = ImageTk.PhotoImage(Image.fromarray(rgb))

        rst.config(image=imgTk)
        rst.image = imgTk
        rst.place(x=50, y=40)
        imageFrame.place(x=250, y=120)

        try:
            window.update()
        except:
            break

    cap.release()
    cv.destroyAllWindows()


# --------------------------------------------------------------------
# QUIT WINDOW HANDLER
# --------------------------------------------------------------------
def on_closing():
    from tkinter import messagebox
    if messagebox.askokcancel("Quit", "Do you want to quit?"):
        try:
            cv.destroyAllWindows()
        except:
            pass
        window.destroy()

window.protocol("WM_DELETE_WINDOW", on_closing)


# --------------------------------------------------------------------
# GUI BUTTONS
# --------------------------------------------------------------------
tk.Button(window, text="video", fg="white", bg="lawn green", command=runToTheVideo,
          width=10, height=2, padx=10, pady=10, relief=SUNKEN,
          font=('times', 15, 'bold')).place(x=90, y=350)

tk.Button(window, text="Image", fg="white", bg="lawn green", command=runToTheImage,
          width=10, height=2, padx=10, pady=10, relief=SUNKEN,
          font=('times', 15, 'bold')).place(x=90, y=450)

tk.Button(window, text="Live", fg="white", bg="lawn green", command=runLive,
          width=10, height=2, padx=10, pady=10, relief=SUNKEN,
          font=('times', 15, 'bold')).place(x=90, y=250)

tk.Button(window, text="Quit", fg="white", bg="red", command=on_closing,
          width=10, height=2, padx=10, pady=10, relief=SUNKEN,
          font=('times', 15, 'bold')).place(x=1000, y=250)

window.mainloop()


In [6]:
import numpy as np
import cv2 as cv
import time
import matplotlib.pyplot as plt
from tkinter import *
import tkinter as tk
from PIL import Image, ImageTk
import random   

window = tk.Tk()

# MAIN WINDOW SETTINGS
imgs = tk.PhotoImage(file="C:\\Users\\HP\\Documents\\Object Detection\\Project\\Object Detection In Real Time\\finalize.png")

window.title("OBJECT DETECTION")
window.geometry('1100x650')
window.configure(background='black')

LabelImg = Label(image=imgs)
LabelImg.pack()

message = tk.Label(window, text="OBJECT DETECTION", bg="light blue", fg="black", 
                   width=48, height=2, font=('times', 30, 'italic bold'))
message.place(x=50, y=10)


# --------------------------------------------------------------------
# SAFE WINDOW CHECK (PREVENTS CRASH)
# --------------------------------------------------------------------
def window_alive():
    try:
        return window.winfo_exists() == 1
    except:
        return False


# --------------------------------------------------------------------
# EXIT FUNCTION
# --------------------------------------------------------------------
def exit():
    try:
        cap.release()
    except:
        pass
    
    if window_alive():
        window.destroy()


# --------------------------------------------------------------------
# RUN ON IMAGE
# --------------------------------------------------------------------
def runToTheImage():

    if not window_alive():
        return

    labels = []
    classFile = 'coco.names'
    with open(classFile, 'rt') as f:
        labels = f.read().rstrip('\n').split('\n')

    label_colors = np.random.uniform(0, 255, (len(labels), 3))

    tf_net = cv.dnn.readNetFromTensorflow(
        "frozen_inference_graph.pb",
        "ssd_mobilenet_v3_large_coco_2020_01_14.pbtxt"
    )

    images = [
        "person.jpg", "cats.jpg", "dog.jpg", "electro.png",
        "reading.jpg", "sofa.jpg", "street.jpg", "bus.jpg","Hand.jpg","Finger.jpg","pen.jpg"
    ]

    number = random.randint(0, len(images)-1)
    img = cv.imread(images[number])

    rows, cols = img.shape[:2]

    blob = cv.dnn.blobFromImage(img, 1.0/127.5, (300, 300),
                                (127.5, 127.5, 127.5), swapRB=True, crop=False)
    tf_net.setInput(blob)
    out = tf_net.forward()

    confidence = 0.5

    for detection in out[0, 0, :, :]:
        score = float(detection[2])
        if score > confidence:
            label = int(detection[1]) - 1
            left = int(detection[3] * cols)
            top = int(detection[4] * rows)
            right = int(detection[5] * cols)
            bottom = int(detection[6] * rows)

            text = f"{labels[label]} - {round(score*100, 2)}%"
            cv.putText(img, text, (left, top), cv.FONT_HERSHEY_SIMPLEX,
                       0.5, label_colors[label], 2)
            cv.rectangle(img, (left, top), (right, bottom),
                         label_colors[label], 3)

    if not window_alive():
        return

    imageFrame = Frame(window, bg="grey", width=700, height=600)
    rst = tk.Label(imageFrame, background="snow", fg="black", font=("", 15))

    out_img = cv.resize(img, (600, 500))
    rgb = cv.cvtColor(out_img, cv.COLOR_BGR2RGB)
    imgTk = ImageTk.PhotoImage(Image.fromarray(rgb))

    rst.config(image=imgTk)
    rst.image = imgTk
    rst.place(x=50, y=40)

    imageFrame.place(x=250, y=120)

    def clear():
        imageFrame.destroy()

    Button(window, text="Clear", command=clear, fg="black", bg="lawn green",
           padx=10, pady=10, width=10, height=2, font=('times', 15, 'bold')).place(x=1000, y=350)


# --------------------------------------------------------------------
# RUN ON VIDEO
# --------------------------------------------------------------------
def runToTheVideo():

    if not window_alive():
        return

    labels = []
    classFile = 'coco.names'
    with open(classFile, 'rt') as f:
        labels = f.read().rstrip('\n').split('\n')

    label_colors = np.random.uniform(0, 255, (len(labels), 3))

    tf_net = cv.dnn.readNetFromTensorflow(
        "frozen_inference_graph.pb",
        "ssd_mobilenet_v3_large_coco_2020_01_14.pbtxt"
    )

    cap = cv.VideoCapture("video1.mp4")
    pause = True

    while pause:

        if not window_alive():
            cap.release()
            return

        succ, img = cap.read()
        if not succ:
            break

        img = cv.flip(img, 1)

        rows, cols = img.shape[:2]

        blob = cv.dnn.blobFromImage(img, 1.0/127.5, (300, 300),
                                    (127.5, 127.5, 127.5), swapRB=True, crop=False)

        tf_net.setInput(blob)
        out = tf_net.forward()

        confidence = 0.5

        for detection in out[0, 0, :, :]:
            score = float(detection[2])
            if score > confidence:
                label = int(detection[1]) - 1

                left = int(detection[3] * cols)
                top = int(detection[4] * rows)
                right = int(detection[5] * cols)
                bottom = int(detection[6] * rows)

                text = f"{labels[label]} - {round(score*100, 2)}%"
                cv.putText(img, text, (left, top), cv.FONT_HERSHEY_SIMPLEX,
                           0.6, label_colors[label], 2)
                cv.rectangle(img, (left, top), (right, bottom),
                             label_colors[label], 2)

        if not window_alive():
            cap.release()
            return

        imageFrame = Frame(window, bg="sky blue", width=700, height=600)
        rst = tk.Label(imageFrame, background="snow", fg="black")

        out_img = cv.resize(img, (600, 500))
        rgb = cv.cvtColor(out_img, cv.COLOR_BGR2RGB)
        imgTk = ImageTk.PhotoImage(Image.fromarray(rgb))

        rst.config(image=imgTk)
        rst.image = imgTk
        rst.place(x=50, y=40)
        imageFrame.place(x=250, y=120)

        try:
            window.update()
        except:
            break

    cap.release()
    cv.destroyAllWindows()


# --------------------------------------------------------------------
# RUN LIVE CAMERA
# --------------------------------------------------------------------
def runLive():

    if not window_alive():
        return

    labels = []
    classFile = 'coco.names'
    with open(classFile, 'rt') as f:
        labels = f.read().rstrip('\n').split('\n')

    label_colors = np.random.uniform(0, 255, (len(labels), 3))

    tf_net = cv.dnn.readNetFromTensorflow(
        "frozen_inference_graph.pb",
        "ssd_mobilenet_v3_large_coco_2020_01_14.pbtxt"
    )

    cap = cv.VideoCapture(0)
    pause = True

    while pause:

        if not window_alive():
            cap.release()
            return

        succ, img = cap.read()
        if not succ:
            break

        img = cv.flip(img, 1)
        rows, cols = img.shape[:2]

        blob = cv.dnn.blobFromImage(img, 1.0/127.5, (300, 300),
                                    (127.5, 127.5, 127.5), swapRB=True, crop=False)
        tf_net.setInput(blob)
        out = tf_net.forward()

        confidence = 0.7

        for detection in out[0, 0, :, :]:
            score = float(detection[2])
            if score > confidence:
                label = int(detection[1]) - 1

                left = int(detection[3] * cols)
                top = int(detection[4] * rows)
                right = int(detection[5] * cols)
                bottom = int(detection[6] * rows)

                text = f"{labels[label]} - {round(score*100, 2)}%"
                cv.putText(img, text, (left, top), cv.FONT_HERSHEY_SIMPLEX,
                           0.5, label_colors[label], 2)
                cv.rectangle(img, (left, top), (right, bottom),
                             label_colors[label], 3)

        if not window_alive():
            cap.release()
            return

        imageFrame = Frame(window, bg="sky blue", width=700, height=600)
        rst = tk.Label(imageFrame, background="snow", fg="black")

        out_img = cv.resize(img, (600, 500))
        rgb = cv.cvtColor(out_img, cv.COLOR_BGR2RGB)
        imgTk = ImageTk.PhotoImage(Image.fromarray(rgb))

        rst.config(image=imgTk)
        rst.image = imgTk
        rst.place(x=50, y=40)
        imageFrame.place(x=250, y=120)

        try:
            window.update()
        except:
            break

    cap.release()
    cv.destroyAllWindows()


# --------------------------------------------------------------------
# QUIT WINDOW HANDLER
# --------------------------------------------------------------------
def on_closing():
    from tkinter import messagebox
    if messagebox.askokcancel("Quit", "Do you want to quit?"):
        try:
            cv.destroyAllWindows()
        except:
            pass
        window.destroy()

window.protocol("WM_DELETE_WINDOW", on_closing)


# --------------------------------------------------------------------
# GUI BUTTONS
# --------------------------------------------------------------------
tk.Button(window, text="video", fg="white", bg="lawn green", command=runToTheVideo,
          width=10, height=2, padx=10, pady=10, relief=SUNKEN,
          font=('times', 15, 'bold')).place(x=90, y=350)

tk.Button(window, text="Image", fg="white", bg="lawn green", command=runToTheImage,
          width=10, height=2, padx=10, pady=10, relief=SUNKEN,
          font=('times', 15, 'bold')).place(x=90, y=450)

tk.Button(window, text="Live", fg="white", bg="lawn green", command=runLive,
          width=10, height=2, padx=10, pady=10, relief=SUNKEN,
          font=('times', 15, 'bold')).place(x=90, y=250)

tk.Button(window, text="Quit", fg="white", bg="red", command=on_closing,
          width=10, height=2, padx=10, pady=10, relief=SUNKEN,
          font=('times', 15, 'bold')).place(x=1000, y=250)

window.mainloop()


Exception in Tkinter callback
Traceback (most recent call last):
  File "C:\Users\HP\anaconda3\lib\tkinter\__init__.py", line 1892, in __call__
    return self.func(*args)
  File "C:\Users\HP\AppData\Local\Temp\ipykernel_12448\28114670.py", line 78, in runToTheImage
    rows, cols = img.shape[:2]
AttributeError: 'NoneType' object has no attribute 'shape'
